# Coffee17 Frozen Representation Geometry Audit V1

Decision experiment: frozen ImageNet MobileNetV3-Large versus frozen DINOv2-S/14 on the same five locked Coffee17 development folds. Raw RGB only; kNN + fixed-C linear probe; outer test untouched.


In [ ]:
NOTEBOOK_BUILD = "COFFEE17-REP-GEOMETRY-V1"
SCIENTIFIC_CODE_COMMIT = "4625381280b04c76d7d4a52ab406c843ede88ab7"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-rep-audit"
PROJECT = WORK / "coffee17-representation-geometry-v1-project"
OUT = PROJECT / "experiments/coffee17-representation-geometry-v1"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

for path in (REPO, PROJECT):
    shutil.rmtree(path, ignore_errors=True)
PROJECT.mkdir(parents=True)
(PROJECT / "logs").mkdir(parents=True)
(PROJECT / "analysis").mkdir(parents=True)
OUT.mkdir(parents=True)

run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

run([sys.executable, "-m", "pip", "install", "-q", "-r",
     REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/experiments/test_representation_geometry_audit.py",
], cwd=REPO)

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_rep_geometry_original.zip"
PROV_LOCAL = WORK / "coffee17_rep_geometry_provenance"
CANONICAL = WORK / "coffee17_rep_geometry_original_v1"
FOLDS_LOCAL = WORK / "coffee17_rep_geometry_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

CONFIG = REPO / "configs/representation_geometry/COFFEE17_REP_GEOMETRY_V1.yaml"

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_rep_geometry_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_representation_geometry_audit",
        "--config", CONFIG,
        "--data-root", DEV,
        "--output-dir", OUT / f"fold_{fold}",
        "--device", "cuda:0",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "representation_geometry_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_representation_geometry_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
cmp = summary["comparison"]["dinov2_minus_mobilenet"]
gate = summary["representation_gate"]

print("\n=== DECISION ===")
print("Gate:", gate["decision"])
print(
    "Linear Macro delta:",
    f"{cmp['linear_probe']['macro_f1']['mean']:+.4%}",
    "| positive folds:",
    cmp["linear_probe"]["macro_f1"]["positive_folds"], "/5",
)
print(
    "Linear Hard delta:",
    f"{cmp['linear_probe']['hard_class_f1']['mean']:+.4%}",
    "| positive folds:",
    cmp["linear_probe"]["hard_class_f1"]["positive_folds"], "/5",
)
print(
    "Audited pair confusions:",
    gate["mobilenet_linear_probe_pair_confusions_total"],
    "->",
    gate["dinov2_linear_probe_pair_confusions_total"],
)
print("Outer test accessed:", summary["outer_test_accessed"])

PKG = WORK / "coffee17-representation-geometry-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copy2(SUMMARY, PKG / SUMMARY.name)
shutil.copy2(CONFIG, PKG / CONFIG.name)
shutil.copy2(
    REPO / "docs/protocols/COFFEE17_REPRESENTATION_GEOMETRY_AUDIT_V1.md",
    PKG / "COFFEE17_REPRESENTATION_GEOMETRY_AUDIT_V1.md",
)

for fold in range(1, 6):
    src = OUT / f"fold_{fold}"
    dst = PKG / f"fold_{fold}"
    dst.mkdir(parents=True)
    shutil.copy2(src / "audit_result.json", dst / "audit_result.json")
    for encoder in ("mobilenetv3_imagenet", "dinov2_small"):
        shutil.copytree(src / encoder, dst / encoder)

zip_path = Path(shutil.make_archive(
    str(WORK / "coffee17-representation-geometry-v1-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
